Calculating the contrast to noise ratio etc.

In [ ]:
# Construct some background and noise ROIs for CNR/CBR calculations.

import patato as pat

import numpy as np
from scipy.stats import pearsonr
import pandas as pd

from matplotlib.transforms import blended_transform_factory
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import copy

import statsmodels.formula.api as smf

from paiskintonetools import setup_matplotlib  #  type:ignore
from paiskintonetools.stats import loess_bootstrap  #  type:ignore
import scienceplots  # noqa

setup_matplotlib(dpi=200)
plt.style.use("high-contrast")

In [2]:
def agg_function(x, npf=np.nanmean):
    if np.issubdtype(x.dtype, np.number):
        return npf(x)
    elif hasattr(x.iloc[0], "dtype"):
        # Assume that this is a numpy array.
        return npf(np.stack(x), axis=0)
    else:
        return x.iloc[0]

In [3]:
import json

settings = json.load(open("../data_paths.json", "r"))
data_path = (Path.cwd() / "../" / settings["main_data_path"]).resolve()


root_path = Path(data_path)

In [4]:
def get_translated_rois(pa, roi_name) -> tuple[pat.ROI, pat.ROI]:
    # Get three copies of the roi to use as templates.
    roi_background = pa.get_rois()[roi_name]
    roi_background.points[:, 0] -= 1e-2

    roi_original = pa.get_rois()[roi_name]

    # roi_noise = pa.get_rois()[roi_name]
    # roi_noise.points = np.array(
    #     [[-0.01, 0.01], [-0.01, 0.015], [0.01, 0.015], [0.01, 0.01], [-0.01, 0.01]]
    # )

    # roi_noise.generated = True
    # roi_noise.roi_class = "background"
    # roi_noise.position = "above"

    return roi_background, roi_original  # , roi_noise

In [5]:
def get_translated_rois_data(pa, roi_name):
    roi_background, roi_signal = get_translated_rois(pa, roi_name)

    rec = pa.get_scan_reconstructions()["Model Based", "0"]
    rec_data = np.squeeze(rec.raw_data)
    # Get the masks.
    mask_background, _ = roi_background.to_mask_slice(rec)  # type: ignore
    mask_signal, _ = roi_signal.to_mask_slice(rec)  # type: ignore

    mask_background = np.squeeze(mask_background)
    mask_signal = np.squeeze(mask_signal)

    results = {}

    # Noise level = standard deviation across the noise mask
    noise_bg_region = np.std(rec_data.T[mask_background.T].T, axis=-1)
    np.std(rec_data.T[mask_signal.T].T, axis=-1)
    noise = noise_bg_region
    results["noise_level"] = noise

    signal = np.mean(rec_data.T[mask_signal.T].T, axis=-1)
    results["signal"] = signal

    background = np.mean(rec_data.T[mask_background.T].T, axis=-1)
    results["background"] = background

    results["CNR"] = np.abs(signal - background) / noise
    results["contrast"] = np.abs(signal - background)  # / noise
    return results


def get_cnr(pa_file):
    pa = pat.PAData.from_hdf5(pa_file)
    if ("artery_", "0") not in pa.get_rois():
        # print(pa_file, pa.get_scan_name())
        return pd.Series(get_translated_rois_data(pa, ("artery_radial", "0")))
    return pd.Series(get_translated_rois_data(pa, ("artery_", "0")))

In [6]:
df_scans = pd.read_parquet("scan_table.parquet")
df_fp = pd.read_excel("../SummaryTables/SummaryDetails.xlsx")
df_scans = pd.merge(df_scans, df_fp)

In [7]:
ra_scans = df_scans.query(
    "Region == 'Radial Artery' & Parallel == '' & `Fitzpatrick Type` != 'Vitiligo'"
).copy()

In [8]:
# Correct the file path:

import json

settings = json.load(open("../data_paths.json", "r"))
root_data_path = (Path.cwd() / "../" / settings["main_data_path"]).resolve()

ra_scans["File"] = ra_scans["File"].apply(
    lambda x: str(root_data_path / Path(x).parent.stem / (Path(x).stem + ".hdf5"))
)

In [9]:
cnr_results = ra_scans["File"].apply(get_cnr)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Volumes/Extreme SSD/Papers/PAISKINTONE/Data/SKIN01/Scan_7.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
wavelengths = pat.PAData.from_hdf5(
    root_data_path / "SKIN14/Scan_31.hdf5"
).get_wavelengths()

In [ ]:
ra_results = pd.concat([ra_scans, cnr_results], axis=1)

In [ ]:
ra_results["cnr700"] = ra_results["CNR"].apply(lambda x: x[0])
ra_results["cnr800"] = ra_results["CNR"].apply(lambda x: x[3])
ra_results["cnr1080"] = ra_results["CNR"].apply(lambda x: x[10])

ra_results["contrast700"] = ra_results["contrast"].apply(lambda x: x[0])
ra_results["contrast800"] = ra_results["contrast"].apply(lambda x: x[3])
ra_results["contrast1080"] = ra_results["contrast"].apply(lambda x: x[10])


ra_results["background700"] = ra_results["background"].apply(lambda x: x[0])
ra_results["background800"] = ra_results["background"].apply(lambda x: x[3])
ra_results["background1080"] = ra_results["background"].apply(lambda x: x[10])


ra_results["signal700"] = ra_results["signal"].apply(lambda x: x[0])
ra_results["signal800"] = ra_results["signal"].apply(lambda x: x[3])
ra_results["signal1080"] = ra_results["signal"].apply(lambda x: x[10])


ra_results["noise700"] = ra_results["noise_level"].apply(lambda x: x[0])
ra_results["noise800"] = ra_results["noise_level"].apply(lambda x: x[3])
ra_results["noise1080"] = ra_results["noise_level"].apply(lambda x: x[10])

In [ ]:
ra_agg = (
    ra_results.groupby(["SkinID", "RunNumber"])
    .agg(agg_function)
    .groupby(level=0)
    .agg(agg_function)
)

ra_agg[
    ["ITA", "signal700", "signal800", "signal1080", "cnr700", "cnr800", "cnr1080"]
].to_csv("../Analysis/plotted_data/figureS7.csv")

In [ ]:
df = pd.read_csv("../Analysis/plotted_data/figureS7.csv")

for column in ["signal700", "signal800", "signal1080", "cnr700", "cnr800", "cnr1080"]:
    model = smf.ols(f"{column} ~ ITA", data=df).fit()
    display(f"column p={model.pvalues['ITA']}")
    display(model.summary())

# Make a figure to demonstrate the ROI positions:

In [ ]:
example_dataset = root_path / "SKIN01/Scan_7.hdf5"
pa = pat.PAData.from_hdf5(example_dataset)  #  type:ignore
print(pa.get_scan_name())
roi_background, roi_original = get_translated_rois(pa, ("artery_", "0"))

# displace the points
fig = plt.figure(figsize=(6, 2))
# sf1, sf2 = fig.subfigures(1, 2)
ax1, ax3, ax2 = fig.subplots(1, 3)
im = pa.get_scan_reconstructions()["Model Based", "0"].imshow(
    ax=ax1,
    scale_kwargs={"length_fraction": 0.3, "font_properties": dict(size="small")},
    clim=(-0.04505299843651125, 0.08144940844742656),
)  #  type:ignore
# print(im.get_clim())
lines = roi_original.plot(linewidth=1, linestyle="--", ax=ax1)
lines[0].set_color("r")
lines[0].set_label("Signal")
lines = roi_background.plot(linewidth=1, linestyle="--", ax=ax1)
lines[0].set_color("g")
lines[0].set_label("Background")

fig.legend(*ax1.get_legend_handles_labels(), loc="lower left")
# ax1.set_title("A", loc="left", fontweight="bold", fontsize="large", ha="left", va="top")
# ax2.set_title("B", loc="left", fontweight="bold", fontsize="large", ha="left", va="top")

# Now add the CNR calculations and CBR.

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3), sharex=True)
sns.regplot(
    data=ra_agg,
    x="ITA",
    y="signal700",
    ax=ax3,
    # lowess=True,
    label="700 nm",
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)

model = smf.ols("signal700 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax3.text(10, 0.005, p_text, c="C0", fontweight="bold")

sns.regplot(
    data=ra_agg,
    x="ITA",
    y="signal800",
    ax=ax3,
    # lowess=True,
    label="800 nm",
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)

model = smf.ols("signal800 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax3.text(0, 0.028, p_text, c="C1", fontweight="bold")

sns.regplot(
    data=ra_agg,
    x="ITA",
    y="signal1080",
    ax=ax3,
    # lowess=True,
    label="1080 nm",
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)

model = smf.ols("signal1080 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax3.text(50, 0.01, p_text, c="C2", fontweight="bold")

ax3.set_xlabel("ITA (degrees)")
ax3.set_ylabel("PA mean (arb. units)")
ax3.invert_xaxis()
ax3.set_title(
    "*Including negative pixels",
    loc="left",
    fontweight="normal",
    fontsize="small",
)
ax3.set_ylim([0, None])


handles, labels = ax3.get_legend_handles_labels()
new_handles = []
for i, h in enumerate(handles):
    new_handle = copy.copy(h)
    new_handle.set_alpha(1)
    new_handle.set_sizes([10])
    new_handles.append(new_handle)
fig.legend(new_handles, labels, loc="outside center right", title="Wavelength")

sns.regplot(
    data=ra_agg,
    x="ITA",
    y="cnr700",
    ax=ax2,
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)
model = smf.ols("cnr700 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax2.text(-10, 0.3, p_text, c="C0", ha="right", fontweight="bold")

sns.regplot(
    data=ra_agg,
    x="ITA",
    y="cnr800",
    ax=ax2,
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)

model = smf.ols("cnr800 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax2.text(55, 18, p_text, c="C1", fontweight="bold")

sns.regplot(
    data=ra_agg,
    x="ITA",
    y="cnr1080",
    ax=ax2,
    scatter_kws=dict(s=5, alpha=0.3, linewidths=0),
)

model = smf.ols("cnr1080 ~ ITA", data=ra_agg).fit()
p = model.pvalues["ITA"]
if p < 0.001:
    p_text = "p < 0.001"
elif p > 0.1:
    p_text = "n.s."
else:
    p_text = f"p = {p:.3f}"
ax2.text(0, 12, p_text, c="C2", fontweight="bold")

ax2.set_xlabel("ITA (degrees)")
ax2.set_ylabel("Contrast to noise ratio")


transform = blended_transform_factory(ax1.transAxes, fig.transFigure)
fig.text(
    0,
    1,
    "A",
    va="top",
    ha="left",
    fontsize="large",
    fontweight="bold",
    transform=transform,
)
transform = blended_transform_factory(ax3.transAxes, fig.transFigure)
fig.text(
    -0.25,
    1,
    "B",
    va="top",
    ha="right",
    fontsize="large",
    fontweight="bold",
    transform=transform,
)
ax2.invert_xaxis()

transform = blended_transform_factory(ax2.transAxes, fig.transFigure)
fig.text(
    -0.2,
    1,
    "C",
    va="top",
    ha="right",
    fontsize="large",
    fontweight="bold",
    transform=transform,
)
fig.savefig("figures/cnr.pdf", dpi=300)
plt.show()

In [ ]:
itas = ra_agg["ITA"].values
cnrs = np.stack(ra_agg["CNR"].values)  # type: ignore
gradients = []
cnr_low_ita = []
cnr_low_ita_upper = []
cnr_low_ita_lower = []
cnr_high_ita = []
cnr_high_ita_upper = []
cnr_high_ita_lower = []

cnr_mid_ita = []
cnr_mid_ita_upper = []
cnr_mid_ita_lower = []

cmap = plt.colormaps["viridis"]
for i in range(cnrs.shape[1]):
    # lm = smf.ols("y~x", data={"x": itas, "y": cnrs[:, i]}).fit()
    # ita_range = np.linspace(np.min(itas), np.max(itas))  # type:ignore
    # gradients.append(lm.pvalues["x"])
    _, mean, upper, lower = loess_bootstrap(
        itas, cnrs[:, i], x_eval=np.array([-40, 10, 60])
    )
    # Low ITA

    cnr_low_ita.append(mean[0])
    cnr_low_ita_lower.append(lower[0])
    cnr_low_ita_upper.append(upper[0])

    cnr_mid_ita.append(mean[1])
    cnr_mid_ita_lower.append(lower[1])
    cnr_mid_ita_upper.append(upper[1])

    # Low ITA
    cnr_high_ita.append(mean[2])
    cnr_high_ita_lower.append(lower[2])
    cnr_high_ita_upper.append(upper[2])
#     plt.plot(ita_range, lm.predict({"x": ita_range}), c=cmap(i / (cnrs.shape[1] - 1)))
# plt.gca().invert_xaxis()
# plt.show()

In [ ]:
lower, mean

In [ ]:
fig, (ax2) = plt.subplots(1, 1, figsize=(3, 2))
# ax1.plot(wavelengths[:-1], -np.log(gradients)[:-1], c="k")
# ax1.axhline(-np.log(0.05), c="k", linestyle="--")
# ax1.set_xlabel("Wavelength (nm)")
# ax1.set_ylabel("-log(p-value)")

lines = ax2.plot(
    wavelengths[:-1], cnr_low_ita[:-1], label="$-$40", c=sns.color_palette("tab10")[0]
)
ax2.fill_between(
    wavelengths[:-1],
    cnr_low_ita_lower[:-1],
    cnr_low_ita_upper[:-1],
    facecolor=lines[0].get_color(),
    alpha=0.3,
)

lines = ax2.plot(
    wavelengths[:-1], cnr_mid_ita[:-1], label="10", c=sns.color_palette("tab10")[1]
)
ax2.fill_between(
    wavelengths[:-1],
    cnr_mid_ita_lower[:-1],
    cnr_mid_ita_upper[:-1],
    facecolor=lines[0].get_color(),
    alpha=0.3,
)


lines = ax2.plot(
    wavelengths[:-1], cnr_high_ita[:-1], label="60", c=sns.color_palette("tab10")[2]
)
ax2.fill_between(
    wavelengths[:-1],
    cnr_high_ita_lower[:-1],
    cnr_high_ita_upper[:-1],
    facecolor=lines[0].get_color(),
    alpha=0.3,
)

ax2.axhline(1, c="k", linestyle="--")
ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel("CNR")
fig.legend(
    *ax2.get_legend_handles_labels(),
    title=r"ITA ($\degree$)",
    loc="outside center right",
)
plt.savefig("figures/cnr_ita_wavelength.pdf", dpi=300)
plt.show()

In [ ]:
# Loop through all wavelengths and then show the statistical dependence on ITA
ps = []
for i, wl in enumerate(wavelengths):
    ra_results["cnrwavelength"] = ra_results["CNR"].apply(lambda x: x[i])
    ra_agg = (
        ra_results.groupby(["SkinID", "RunNumber"])
        .agg(agg_function)
        .groupby(level=0)
        .agg(agg_function)
    )

    sr = pearsonr(ra_agg["cnrwavelength"], ra_agg["ITA"])

    model = smf.ols("cnrwavelength ~ ITA", data=ra_agg).fit()
    p = model.pvalues["ITA"]
    ps.append(sr.pvalue)
print(list(zip(wavelengths, ps)))
plt.semilogy(wavelengths, ps)
plt.axhline(0.05, c="k", linestyle="--")
plt.xlabel("Wavelength (nm)")
plt.ylabel("p-value")
plt.show()